In [2]:
# ============================================================
# KNN CLASSIFICATION
# DATASET UPLOAD + TARGET SELECTION + K SELECTION + ANIMATION
# GOOGLE COLAB VERSION
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import io
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import ipywidgets as widgets

# Maximum size of embedded animation = 100 MB
mpl.rcParams["animation.embed_limit"] = 100

# ============================================================
# 2. TITLE
# ============================================================

print("=" * 60)
print("                  KNN CLASSIFICATION")
print("=" * 60)

print("\nUpload your dataset.")
print("Supported formats: CSV, XLSX, XLS")

# ============================================================
# 3. UPLOAD DATASET
# ============================================================

try:

    from google.colab import files

    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file was uploaded.")

    filename = list(uploaded.keys())[0]

except ImportError:

    raise RuntimeError(
        "This program is designed for Google Colab. "
        "Please run it in Google Colab."
    )

print("\nUploaded file:", filename)

# ============================================================
# 4. READ DATASET
# ============================================================

try:

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if filename.lower().endswith((".csv", ".txt")):

        try:

            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python"
            )

        except UnicodeDecodeError:

            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python",
                encoding="latin1"
            )

    # --------------------------------------------------------
    # EXCEL
    # --------------------------------------------------------

    elif filename.lower().endswith((".xlsx", ".xls")):

        df = pd.read_excel(
            io.BytesIO(uploaded[filename])
        )

    # --------------------------------------------------------
    # UNSUPPORTED FILE
    # --------------------------------------------------------

    else:

        raise ValueError(
            "Unsupported file format. "
            "Please upload CSV, XLSX or XLS."
        )

except Exception as e:

    print("\nERROR READING DATASET")
    print("---------------------")
    print(e)

    raise

# ============================================================
# 5. CLEAN DATASET
# ============================================================

# Remove completely empty rows
df = df.dropna(
    axis=0,
    how="all"
)

# Remove completely empty columns
df = df.dropna(
    axis=1,
    how="all"
)

# Clean column names
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

# ============================================================
# 6. CHECK DATASET
# ============================================================

if df.empty:

    raise ValueError(
        "The uploaded dataset is empty."
    )

if len(df.columns) < 3:

    raise ValueError(
        "The dataset must contain at least "
        "3 columns: two features and one target."
    )

# ============================================================
# 7. DISPLAY DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("\nNumber of Rows    :", df.shape[0])
print("Number of Columns :", df.shape[1])

print("\nColumn Names:")

for i, column in enumerate(df.columns):

    print(
        f"{i + 1}. {column}"
    )

print("\nFirst 5 Rows:")

display(df.head())

# ============================================================
# 8. DETERMINE NUMERICAL COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("\n" + "=" * 60)
print("NUMERICAL COLUMNS")
print("=" * 60)

if len(numeric_columns) == 0:

    raise ValueError(
        "No numerical columns were found. "
        "KNN visualization requires numerical features."
    )

for column in numeric_columns:

    print("•", column)

# ============================================================
# 9. TARGET COLUMN SELECTION
# ============================================================

print("\n" + "=" * 60)
print("SELECT TARGET COLUMN")
print("=" * 60)

# Automatically select Purchased if it exists.
# Otherwise select the last column.

if "Purchased" in df.columns:

    default_target = "Purchased"

else:

    default_target = df.columns[-1]

target_dropdown = widgets.Dropdown(

    options=list(df.columns),

    value=default_target,

    description="Target:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)

display(target_dropdown)

# ============================================================
# 10. K VALUE SELECTION
# ============================================================

print("\nSelect the value of K:")

k_slider = widgets.IntSlider(

    value=3,

    min=1,

    max=15,

    step=1,

    description="K:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)

display(k_slider)

# ============================================================
# 11. RUN BUTTON
# ============================================================

run_button = widgets.Button(

    description="Run KNN",

    button_style="success",

    icon="play",

    layout=widgets.Layout(

        width="200px",

        height="40px"
    )
)

display(run_button)

# ============================================================
# 12. OUTPUT AREA
# ============================================================

output_area = widgets.Output()

display(output_area)

# ============================================================
# 13. KNN FUNCTION
# ============================================================

def run_knn(button):

    # Clear previous output
    with output_area:

        output_area.clear_output(wait=True)

        # ====================================================
        # GET USER SELECTIONS
        # ====================================================

        target_column = target_dropdown.value

        K = k_slider.value

        print("\n")

        print("=" * 60)
        print("                  KNN ANALYSIS")
        print("=" * 60)

        print(
            "\nTarget Column :",
            target_column
        )

        print(
            "K             :",
            K
        )

        # ====================================================
        # 14. IDENTIFY NUMERICAL FEATURES
        # ====================================================

        numeric_columns = df.select_dtypes(
            include=np.number
        ).columns.tolist()

        feature_columns = [

            column

            for column in numeric_columns

            if column != target_column

        ]

        # ====================================================
        # 15. REQUIRE EXACTLY TWO FEATURES
        # ====================================================

        if len(feature_columns) < 2:

            print("\nERROR")
            print("-" * 60)

            print(
                "At least TWO numerical feature columns "
                "are required."
            )

            print(
                "\nAvailable numerical features:",
                feature_columns
            )

            return

        # ----------------------------------------------------
        # If more than two numerical features exist,
        # allow the user to select exactly two.
        # ----------------------------------------------------

        if len(feature_columns) > 2:

            print("\n" + "=" * 60)
            print("SELECT TWO FEATURES FOR KNN")
            print("=" * 60)

            print(
                "\nYour dataset contains",
                len(feature_columns),
                "numerical features."
            )

            print(
                "For the animated 2-D visualization, "
                "please select exactly TWO."
            )

            feature_selector = widgets.SelectMultiple(

                options=feature_columns,

                description="Features:",

                rows=min(10, len(feature_columns)),

                style={
                    "description_width": "initial"
                },

                layout=widgets.Layout(
                    width="450px"
                )
            )

            display(feature_selector)

            print(
                "\nPlease select exactly TWO features "
                "and click Run KNN again."
            )

            # Store selector globally for next click
            global selected_feature_selector

            selected_feature_selector = feature_selector

            return

        # ====================================================
        # 16. HANDLE FEATURE SELECTION
        # ====================================================

        if len(feature_columns) == 2:

            selected_features = feature_columns

        else:

            selected_features = list(
                selected_feature_selector.value
            )

            if len(selected_features) != 2:

                print("\nERROR")
                print("-" * 60)

                print(
                    "Please select exactly TWO "
                    "numerical features."
                )

                return

        feature_columns = selected_features

        # ====================================================
        # 17. DISPLAY FEATURES
        # ====================================================

        print("\nFeatures used:")

        for feature in feature_columns:

            print("•", feature)

        # ====================================================
        # 18. CREATE MODEL DATA
        # ====================================================

        data = df[
            feature_columns + [target_column]
        ].copy()

        # ====================================================
        # 19. REMOVE MISSING VALUES
        # ====================================================

        before_rows = len(data)

        data = data.dropna()

        removed_rows = before_rows - len(data)

        if removed_rows > 0:

            print(
                "\nMissing-value rows removed:",
                removed_rows
            )

        # ====================================================
        # 20. CHECK SAMPLE SIZE
        # ====================================================

        if len(data) < 10:

            print("\nERROR")
            print("-" * 60)

            print(
                "Dataset contains too few valid "
                "samples after removing missing values."
            )

            return

        # ====================================================
        # 21. CREATE X AND Y
        # ====================================================

        X = data[
            feature_columns
        ]

        y = data[
            target_column
        ]

        # ====================================================
        # 22. ENCODE TARGET
        # ====================================================

        label_encoder = LabelEncoder()

        y_encoded = label_encoder.fit_transform(
            y.astype(str)
        )

        print("\n" + "=" * 60)
        print("CLASS INFORMATION")
        print("=" * 60)

        print("\nClasses:")

        for i, class_name in enumerate(
            label_encoder.classes_
        ):

            print(
                f"{i} → {class_name}"
            )

        # ====================================================
        # 23. CLASS DISTRIBUTION
        # ====================================================

        unique_classes, class_counts = np.unique(
            y_encoded,
            return_counts=True
        )

        print("\nClass Distribution:")

        for class_id, count in zip(
            unique_classes,
            class_counts
        ):

            print(
                f"{label_encoder.classes_[class_id]} : {count}"
            )

        # ====================================================
        # 24. CHECK NUMBER OF CLASSES
        # ====================================================

        if len(unique_classes) < 2:

            print("\nERROR")
            print("-" * 60)

            print(
                "Target column must contain "
                "at least TWO classes."
            )

            return

        # ====================================================
        # 25. TRAIN / TEST SPLIT
        # ====================================================

        try:

            X_train, X_test, y_train, y_test = train_test_split(

                X,

                y_encoded,

                test_size=0.20,

                random_state=42,

                stratify=y_encoded

            )

        except ValueError:

            print(
                "\nWarning: Stratified split could not "
                "be performed."
            )

            print(
                "Using a regular random split instead."
            )

            X_train, X_test, y_train, y_test = train_test_split(

                X,

                y_encoded,

                test_size=0.20,

                random_state=42

            )

        # ====================================================
        # 26. FEATURE SCALING
        # ====================================================

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        )

        X_test_scaled = scaler.transform(
            X_test
        )

        # ====================================================
        # 27. CHECK K
        # ====================================================

        if K > len(X_train_scaled):

            print("\nERROR")
            print("-" * 60)

            print(
                f"K = {K} is greater than the "
                f"number of training samples "
                f"({len(X_train_scaled)})."
            )

            print(
                f"\nPlease choose K <= "
                f"{len(X_train_scaled)}."
            )

            return

        # ====================================================
        # 28. CREATE KNN MODEL
        # ====================================================

        model = KNeighborsClassifier(

            n_neighbors=K,

            metric="euclidean"
        )

        # ====================================================
        # 29. TRAIN MODEL
        # ====================================================

        model.fit(

            X_train_scaled,

            y_train
        )

        # ====================================================
        # 30. PREDICT
        # ====================================================

        y_pred = model.predict(

            X_test_scaled
        )

        # ====================================================
        # 31. ACCURACY
        # ====================================================

        accuracy = accuracy_score(

            y_test,

            y_pred
        )

        # ====================================================
        # 32. RESULTS
        # ====================================================

        print("\n")

        print("=" * 60)
        print("                  RESULTS")
        print("=" * 60)

        print(
            "\nTotal Samples       :",
            len(data)
        )

        print(
            "Training Samples    :",
            len(X_train)
        )

        print(
            "Testing Samples     :",
            len(X_test)
        )

        print(
            "Number of Features  :",
            len(feature_columns)
        )

        print(
            "K                   :",
            K
        )

        print(
            "\nAccuracy            :",
            f"{accuracy * 100:.2f}%"
        )

        # ====================================================
        # 33. CONFUSION MATRIX
        # ====================================================

        all_labels = np.arange(
            len(label_encoder.classes_)
        )

        cm = confusion_matrix(

            y_test,

            y_pred,

            labels=all_labels
        )

        print("\nConfusion Matrix:")

        print(cm)

        # ====================================================
        # 34. CLASSIFICATION REPORT
        # ====================================================

        print("\nClassification Report:")

        print(

            classification_report(

                y_test,

                y_pred,

                labels=all_labels,

                target_names=[

                    str(x)

                    for x in label_encoder.classes_

                ],

                zero_division=0
            )
        )

        # ====================================================
        # 35. ANIMATION
        # ====================================================

        print("\n")

        print("=" * 60)
        print("           STARTING KNN ANIMATION")
        print("=" * 60)

        # ====================================================
        # 36. USE BOTH FEATURES
        # ====================================================

        # Because we require exactly two features,
        # the visualization and actual KNN model
        # operate in exactly the same feature space.

        X_visual = X_train_scaled

        X_test_visual = X_test_scaled

        # ====================================================
        # 37. SELECT FIRST TEST POINT
        # ====================================================

        test_point = X_test_visual[0]

        actual_class = y_test[0]

        predicted_class = model.predict(

            test_point.reshape(
                1,
                -1
            )

        )[0]

        # ====================================================
        # 38. CALCULATE EUCLIDEAN DISTANCES
        # ====================================================

        distances = np.sqrt(

            np.sum(

                (
                    X_visual -
                    test_point
                ) ** 2,

                axis=1
            )
        )

        # ====================================================
        # 39. SORT DISTANCES
        # ====================================================

        sorted_indices = np.argsort(

            distances
        )

        # ====================================================
        # 40. FIND K NEAREST NEIGHBOURS
        # ====================================================

        nearest_indices = sorted_indices[:K]

        # ====================================================
        # 41. LIMIT DISTANCE ANIMATION FRAMES
        # ====================================================

        max_distance_frames = 30

        total_training_points = len(
            X_visual
        )

        if total_training_points <= max_distance_frames:

            animation_indices = np.arange(
                total_training_points
            )

        else:

            animation_indices = np.linspace(

                0,

                total_training_points - 1,

                max_distance_frames,

                dtype=int
            )

        # ====================================================
        # 42. CREATE FIGURE
        # ====================================================

        fig, ax = plt.subplots(

            figsize=(9, 6)
        )

        # ====================================================
        # 43. PLOT TRAINING CLASSES
        # ====================================================

        classes = np.unique(

            y_train
        )

        for class_value in classes:

            points = X_visual[

                y_train == class_value
            ]

            ax.scatter(

                points[:, 0],

                points[:, 1],

                s=70,

                label=(

                    "Class "

                    +

                    str(

                        label_encoder.classes_[

                            class_value
                        ]

                    )
                )
            )

        # ====================================================
        # 44. PLOT TEST POINT
        # ====================================================

        ax.scatter(

            test_point[0],

            test_point[1],

            marker="*",

            s=350,

            edgecolors="black",

            linewidth=2,

            label="Test Point"
        )

        # ====================================================
        # 45. AXIS LABELS
        # ====================================================

        ax.set_xlabel(

            feature_columns[0],

            fontsize=12
        )

        ax.set_ylabel(

            feature_columns[1],

            fontsize=12
        )

        # ====================================================
        # 46. TITLE
        # ====================================================

        ax.set_title(

            "KNN Classification - Animated",

            fontsize=16
        )

        # ====================================================
        # 47. GRID
        # ====================================================

        ax.grid(

            True,

            alpha=0.3
        )

        # ====================================================
        # 48. LEGEND
        # ====================================================

        ax.legend()

        # ====================================================
        # 49. INFORMATION BOX
        # ====================================================

        info = ax.text(

            0.02,

            0.97,

            "",

            transform=ax.transAxes,

            verticalalignment="top",

            fontsize=10,

            bbox=dict(

                boxstyle="round",

                facecolor="white",

                alpha=0.9
            )
        )

        # ====================================================
        # 50. STORE ANIMATION LINES
        # ====================================================

        lines = []

        # ====================================================
        # 51. ANIMATION UPDATE FUNCTION
        # ====================================================

        def update(frame):

            # ------------------------------------------------
            # Remove previous lines
            # ------------------------------------------------

            for line in lines:

                line.remove()

            lines.clear()

            # =================================================
            # STEP 1
            # =================================================

            if frame == 0:

                info.set_text(

                    "STEP 1\n\n"

                    "Test point selected\n"

                    "Waiting to calculate distances..."
                )

            # =================================================
            # STEP 2
            # DISTANCE CALCULATION
            # =================================================

            elif 1 <= frame <= len(
                animation_indices
            ):

                index = animation_indices[
                    frame - 1
                ]

                point = X_visual[
                    index
                ]

                line, = ax.plot(

                    [

                        test_point[0],

                        point[0]

                    ],

                    [

                        test_point[1],

                        point[1]

                    ],

                    linestyle="--",

                    linewidth=1
                )

                lines.append(line)

                info.set_text(

                    "STEP 2: DISTANCE CALCULATION\n\n"

                    f"Training Point : {index + 1}\n"

                    f"Distance       : "

                    f"{distances[index]:.3f}"
                )

            # =================================================
            # STEP 3
            # K NEAREST NEIGHBOURS
            # =================================================

            elif frame == len(
                animation_indices
            ) + 1:

                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]

                        ],

                        [

                            test_point[1],

                            point[1]

                        ],

                        linewidth=3
                    )

                    lines.append(line)

                neighbour_classes = y_train[
                    nearest_indices
                ]

                neighbour_names = [

                    label_encoder.classes_[c]

                    for c in neighbour_classes

                ]

                info.set_text(

                    "STEP 3: K NEAREST NEIGHBOURS\n\n"

                    f"K = {K}\n"

                    f"Nearest Points = "

                    f"{nearest_indices + 1}\n"

                    f"Classes = "

                    f"{neighbour_names}"
                )

            # =================================================
            # STEP 4
            # MAJORITY VOTING
            # =================================================

            elif frame == len(
                animation_indices
            ) + 2:

                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]

                        ],

                        [

                            test_point[1],

                            point[1]

                        ],

                        linewidth=3
                    )

                    lines.append(line)

                neighbour_classes = y_train[
                    nearest_indices
                ]

                unique, counts = np.unique(

                    neighbour_classes,

                    return_counts=True
                )

                voting_text = ""

                for c, count in zip(

                    unique,

                    counts
                ):

                    voting_text += (

                        f"Class "

                        f"{label_encoder.classes_[c]}"

                        f" → {count} vote(s)\n"
                    )

                info.set_text(

                    "STEP 4: MAJORITY VOTING\n\n"

                    +

                    voting_text
                )

            # =================================================
            # STEP 5
            # FINAL PREDICTION
            # =================================================

            else:

                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]

                        ],

                        [

                            test_point[1],

                            point[1]

                        ],

                        linewidth=3
                    )

                    lines.append(line)

                info.set_text(

                    "STEP 5: FINAL PREDICTION\n\n"

                    f"K = {K}\n"

                    f"Actual Class    : "

                    f"{label_encoder.classes_[actual_class]}\n"

                    f"Predicted Class : "

                    f"{label_encoder.classes_[predicted_class]}"
                )

            return lines + [info]

        # ====================================================
        # 52. CREATE ANIMATION
        # ====================================================

        total_frames = (

            len(animation_indices)

            + 4
        )

        anim = FuncAnimation(

            fig,

            update,

            frames=total_frames,

            interval=350,

            repeat=True,

            blit=False
        )

        # ====================================================
        # 53. DISPLAY ANIMATION
        # ====================================================

        plt.close(fig)

        animation_html = anim.to_jshtml()

        display(

            HTML(

                animation_html
            )
        )

# ============================================================
# 54. CONNECT BUTTON
# ============================================================

run_button.on_click(

    run_knn
)

# ============================================================
# 55. READY MESSAGE
# ============================================================

print("\nReady!")

print(
    "Select the target column, choose K, "
    "and click 'Run KNN'."
)




                  KNN ANALYSIS

Target Column : Purchased
K             : 3

Features used:
• Age
• AnnualIncome

CLASS INFORMATION

Classes:
0 → No
1 → Yes

Class Distribution:
No : 8
Yes : 12


                  RESULTS

Total Samples       : 20
Training Samples    : 16
Testing Samples     : 4
Number of Features  : 2
K                   : 3

Accuracy            : 100.00%

Confusion Matrix:
[[2 0]
 [0 2]]

Classification Report:
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         2
         Yes       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



           STARTING KNN ANIMATION
